In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv(r"D:\Practice\Numpy\Practice\Dataset\4train.csv")

df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [2]:
#1
"""
1. Create a copy of the Age column as a NumPy array. 
2. Replace missing Age values with the median Age of passengers in the same Pclass. 
3. Store the filled values in a new DataFrame column called AgeFilled. 
4. Calculate the mean of AgeFilled for each Pclass using transform(). 
5. Display the 10 passengers whose age differs most from their class average. 

Required columns: Name, Pclass, AgeFilled, ClassAvgAge, AgeDifference
"""

df['AgeFilled'] = df['Age'].fillna(df.groupby('Pclass')['Age'].transform('median'))

df['ClassAvgAge'] = df.groupby('Pclass')['AgeFilled'].transform('mean')

df['AgeDifference'] = (df['AgeFilled'] - df['ClassAvgAge']).abs()

show = df[['Name','Pclass','AgeFilled','ClassAvgAge','AgeDifference']].sort_values('AgeDifference', ascending = False).head(10)

print(show)

                                     Name  Pclass  AgeFilled  ClassAvgAge  \
851                   Svensson, Mr. Johan       3      74.00    24.824684   
116                  Connors, Mr. Patrick       3      70.50    24.824684   
630  Barkworth, Mr. Algernon Henry Wilson       1      80.00    38.062130   
280                      Duane, Mr. Frank       3      65.00    24.824684   
672           Mitchell, Mr. Henry Michael       2      70.00    29.825163   
483                Turkula, Mrs. (Hedwig)       3      63.00    24.824684   
305        Allison, Master. Hudson Trevor       1       0.92    38.062130   
326             Nysveen, Mr. Johan Hansen       3      61.00    24.824684   
33                  Wheadon, Mr. Edward H       2      66.00    29.825163   
297          Allison, Miss. Helen Loraine       1       2.00    38.062130   

     AgeDifference  
851      49.175316  
116      45.675316  
630      41.937870  
280      40.175316  
672      40.174837  
483      38.175316  
305  

In [3]:
#2
"""
1. Convert Fare into a NumPy array copy. 
2. Replace missing fares with the median Fare. 
3. Apply min-max normalization : normalizedFare= 𝑥−min(𝑥) / max(𝑥)−min(𝑥) 
4. Print the minimum, maximum, and mean of the normalized array. 
5. Confirm that the original df['Fare'] column was not modified.
"""

fare = df['Fare'].to_numpy().copy()

fare = np.where(np.isnan(fare), np.nanmedian(fare), fare)

normalizedFare = ((fare - fare.min()) /(fare.max() - fare.min()))

print(f"Minimum of normalized array : {normalizedFare.min()}")
print(f"Maximum of normalized array : {normalizedFare.max()}")
print(f"Mean of normalized array : {normalizedFare.mean()}")

Minimum of normalized array : 0.0
Maximum of normalized array : 1.0
Mean of normalized array : 0.06285842768394742


In [4]:
#3
"""
1. Create FamilySize = SibSp + Parch + 1. 
2. Create a FamilyCategory column: 
                    o Alone: FamilySize = 1 
                    o Small: FamilySize = 2–4 
                    o Large: FamilySize > 4 

3. Calculate the survival rate percentage for each category. 
4. Sort the results from highest to lowest survival rate. 
5. Display the number of passengers in each category. 
"""

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df['FamilyCategory'] = np.where(df['FamilySize'] == 1, 'Alone',
                                    np.where(df['FamilySize'] <= 4,'Small','Large'))

show = (df.groupby('FamilyCategory').agg(
          PassengerCount=('Survived', 'count'),
          SurvivalRate=('Survived', 'mean')
      )
)

show['SurvivalRate'] = show['SurvivalRate'] * 100

show = show.sort_values('SurvivalRate', ascending=False)

print(show)

                PassengerCount  SurvivalRate
FamilyCategory                              
Small                      292     57.876712
Alone                      537     30.353818
Large                       62     16.129032


In [5]:
#4
"""
Use NumPy to detect fare outliers using the IQR method. 
1. Calculate Q1, Q3, and IQR for Fare. 
2. Calculate the lower and upper bounds. 
3. Idetify outliers outside those bounds. 
4. Print the number of outliers and their percentage of the dataset. 
5. Display the 10 highest-fare outliers with Name, Pclass, and Fare.
"""

q1 = np.percentile(fare, 25)
q3 = np.percentile(fare, 75)

iqr = q3 - q1

lower = q1 - (1.5 * iqr)
upper = q3 + (1.5 * iqr)

outlier_mask = (fare < lower) | (fare > upper)

count = np.sum(outlier_mask)
percentage = (count / len(fare)) * 100

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower)
print("Upper bound:", upper)
print("Number of outliers:", count)
print("Percentage of outliers:", percentage)

Q1: 7.9104
Q3: 31.0
IQR: 23.0896
Lower bound: -26.724
Upper bound: 65.6344
Number of outliers: 116
Percentage of outliers: 13.019079685746352


In [6]:
#5
"""
1. Calculate PclassAvgFare using groupby() and transform('mean'). 
2. Create FareRatio = Fare / PclassAvgFare. 
3. Create FareDifference = Fare - PclassAvgFare. 
4. Display the 10 passengers with the highest FareRatio. 
5. Explain what a FareRatio greater than 1 means.
"""

df['PclassAvgFare'] = df.groupby('Pclass')['Fare'].transform('mean')

df['FareRatio'] = df['Fare'] / df['PclassAvgFare']

df['FareDifference'] = df['Fare'] - df['PclassAvgFare']

show = df.sort_values('FareRatio', ascending = False)[['Name', 'Pclass', 'Fare', 'PclassAvgFare','FareRatio', 'FareDifference']].head(10)

print(show)

                                   Name  Pclass      Fare  PclassAvgFare  \
679  Cardeza, Mr. Thomas Drake Martinez       1  512.3292      84.154687   
737              Lesurer, Mr. Gustave J       1  512.3292      84.154687   
258                    Ward, Miss. Anna       1  512.3292      84.154687   
180        Sage, Miss. Constance Gladys       3   69.5500      13.675550   
324            Sage, Mr. George John Jr       3   69.5500      13.675550   
159          Sage, Master. Thomas Henry       3   69.5500      13.675550   
792             Sage, Miss. Stella Anna       3   69.5500      13.675550   
201                 Sage, Mr. Frederick       3   69.5500      13.675550   
863   Sage, Miss. Dorothy Edith "Dolly"       3   69.5500      13.675550   
846            Sage, Mr. Douglas Bullen       3   69.5500      13.675550   

     FareRatio  FareDifference  
679   6.087946      428.174512  
737   6.087946      428.174512  
258   6.087946      428.174512  
180   5.085719       55.874450 

In [7]:
#6
"""
1. Create a pivot table with Pclass as rows and Sex as columns. 
2. Calculate survival percentages for each combination. 
3. Display passenger counts in a second pivot table. 
4. Identify the combination with the highest survival rate using stack().idxmax(). 
5. Explain why the highest survival rate does not necessarily mean the group had the most survivors.
"""

pvt = df.pivot_table(
                        index = 'Pclass',
                        columns = 'Sex',
                        values = 'Survived',
                        aggfunc = 'mean'
                    ) * 100

pvt2 = df.pivot_table(
                        index = 'Pclass',
                        columns = 'Sex',
                        values = 'Survived',
                        aggfunc = 'count'
                    )

high = pvt.stack().idxmax()
highest_rate = pvt.stack().max()

print("Survival percentages:")
print(pvt.round(2))

print("\nPassenger counts:")
print(pvt2)

print(f"\nHighest survival rate: {high}")
print(f"Highest survival percentage: {highest_rate:.2f}%")

Survival percentages:
Sex     female   male
Pclass               
1        96.81  36.89
2        92.11  15.74
3        50.00  13.54

Passenger counts:
Sex     female  male
Pclass              
1           94   122
2           76   108
3          144   347

Highest survival rate: (np.int64(1), 'female')
Highest survival percentage: 96.81%


In [8]:
#7
"""
Using NumPy arrays: 
1. Select passengers with Age >= 18. 
2. From those passengers, select passengers with Fare > 50. 
3. Calculate the percentage of adult passengers whose Fare exceeds 50. 
4. Calculate the mean Fare for this selected group. 
5. Display the number of selected passengers. 

For this question, handle missing Age values using the median before filtering.
"""

age = df['Age'].to_numpy()
fare = df['Fare'].to_numpy().copy()

age = np.where(np.isnan(age), np.nanmedian(age), age)

adult_mask = age >= 18

selected_mask = adult_mask & (fare > 50)

adult_count = adult_mask.sum()
selected_count = selected_mask.sum()

percentage = (selected_count / adult_count) * 100

mean_fare = fare[selected_mask].mean()

print("Selected passengers:", selected_count)
print("Percentage of adults:", percentage)
print("Mean fare:", mean_fare)

Selected passengers: 149
Percentage of adults: 19.151670951156813
Mean fare: 107.85875234899328


In [9]:
#8
"""
1. Fill missing Age values with the overall median. 
2. Calculate the mean Age for each Pclass using transform('mean'). 
3. Calculate each passenger's age difference from their class mean. 
4. Create AgePosition: 
                        o Above Average if the difference is positive 
                        o Below Average if negative 
                        o Average if zero 
5. Display the counts for each AgePosition within each Pclass.
"""

df['Age'] = df['Age'].fillna(df['Age'].median())

df['PclassAge'] = df.groupby('Pclass')['Age'].transform('mean')

df['AgeDiff'] = df['Age'] - df['PclassAge']

df['AgePosition'] = np.where(df['AgeDiff'] > 0, 'Above Average',
                                np.where(df['AgeDiff'] < 0, 'Below Average','Average'))

counts = df.groupby(['Pclass', 'AgePosition']).size().unstack(fill_value=0)

print(counts)

AgePosition  Above Average  Below Average
Pclass                                   
1                       95            121
2                       85             99
3                      297            194


In [20]:
#9
"""
Create these features: 
 FamilySize = SibSp + Parch + 1 
 FarePerPerson = Fare / FamilySize 
 PclassAvgFare using transform('mean') 
 FarePercentOfClassAvg = Fare / PclassAvgFare * 100 
 
Then: 
1. Replace missing Age values with the median Age in a NumPy array. 

2. Create a NumPy matrix with these columns in the exact order: 
Survived, Pclass, Age, FamilySize, FarePerPerson, FarePercentOfClassAvg. 

3. Print the matrix shape. 

4. Print the mean and standard deviation of the FarePerPerson column in the matrix. 

5. Confirm that the matrix has six columns. 

Hint: use np.column_stack().
"""

# Create features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

df['PclassAvgFare'] = df.groupby('Pclass')['Fare'].transform('mean')
df['FarePercentOfClassAvg'] = (df['Fare'] / df['PclassAvgFare']) * 100

# Create NumPy arrays
survived = df['Survived'].to_numpy()
pclass = df['Pclass'].to_numpy()

age = df['Age'].to_numpy().copy()
age = np.where(np.isnan(age), np.nanmedian(age), age)

familysize = df['FamilySize'].to_numpy()
fareperperson = df['FarePerPerson'].to_numpy()
farepercentofclassavg = df['FarePercentOfClassAvg'].to_numpy()

# Create matrix with exactly six columns
mat = np.column_stack((survived,pclass,age,familysize,fareperperson,farepercentofclassavg))

# Matrix information
print("Shape:", mat.shape)
print("Mean FarePerPerson:", np.mean(mat[:, 4]))
print("Standard deviation:", np.std(mat[:, 4]))
print("Has six columns:", mat.shape[1] == 6)

Shape: (891, 6)
Mean FarePerPerson: 19.916375111917638
Standard deviation: 35.82113871450563
Has six columns: True


In [27]:
#10
"""
A travel company wants to study passengers who paid less than the average fare for 
their passenger class. 

1. Create FamilySize. 
2. Create FarePerPerson. 
3. Calculate PclassAvgFare and FarePercentOfClassAvg. 
4. Fill missing Age values with the median in a NumPy array. 
5. Select passengers whose FarePercentOfClassAvg < 100 and whose Age >= 18. 
6. From these passengers, find the five with the highest FarePerPerson. 
7. Display Name, Survived, Pclass, Age, FamilySize, FarePerPerson, and FarePercentOfClassAvg.
"""

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df['FarePerPerson'] = df['Fare'] / df['FamilySize']

df['PclassAvgFare'] = df.groupby('Pclass')['Fare'].transform('mean')

df['FarePercentOfClassAvg'] = (df['Fare'] / df['PclassAvgFare']) * 100

age = df['Age'].to_numpy().copy()
age = np.where(np.isnan(age), np.nanmedian(age), age)

mask = ((df['FarePercentOfClassAvg'] < 100) &(age >= 18))

show = (df.loc[mask, ['Name', 'Survived', 'Pclass', 'Age','FamilySize', 'FarePerPerson','FarePercentOfClassAvg']]
    .sort_values('FarePerPerson', ascending=False)
    .head(5)
)

print(show)

                                          Name  Survived  Pclass   Age  \
310             Hays, Miss. Margaret Bechstein         1       1  24.0   
829  Stone, Mrs. George Nelson (Martha Evelyn)         1       1  62.0   
61                         Icard, Miss. Amelie         1       1  38.0   
789                   Guggenheim, Mr. Benjamin         0       1  46.0   
256             Thorne, Mrs. Gertrude Maybelle         1       1  28.0   

     FamilySize  FarePerPerson  FarePercentOfClassAvg  
310           1        83.1583              98.816005  
829           1        80.0000              95.063035  
61            1        80.0000              95.063035  
789           1        79.2000              94.112405  
256           1        79.2000              94.112405  
